# 03. 오디오 분류 — 멜 스펙트로그램 + CNN

오디오 분류는 소리를 듣고 종류를 맞히는 작업입니다(예: 음악 장르, 환경음, 명령어).
표준 접근은 **오디오를 멜 스펙트로그램(이미지)으로 변환해 CNN으로 분류**하는 것입니다.

이 노트북에서는 개념을 명확히 보기 위해 **합성 오디오**(서로 다른 주파수 패턴)를 만들어 분류합니다.
별도 데이터 다운로드 없이 전체 파이프라인을 경험할 수 있습니다.

1. 합성 오디오 데이터 생성
2. 멜 스펙트로그램 특징 변환
3. CNN 학습 및 평가

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import librosa
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. 합성 오디오 데이터

3개 클래스를 만듭니다: (0) 저주파 톤, (1) 고주파 톤, (2) 두 음의 화음.
각 샘플에 약간의 잡음과 주파수 변동을 주어 다양성을 만듭니다.

In [ ]:
SR = 16000
DUR = 1.0
N_PER_CLASS = 200

def make_sample(kind, rng):
    t = np.linspace(0, DUR, int(SR*DUR), endpoint=False)
    noise = 0.05 * rng.standard_normal(len(t))
    if kind == 0:
        f = rng.uniform(180, 260); sig = np.sin(2*np.pi*f*t)
    elif kind == 1:
        f = rng.uniform(900, 1200); sig = np.sin(2*np.pi*f*t)
    else:
        f1, f2 = rng.uniform(200, 300), rng.uniform(500, 700)
        sig = 0.5*np.sin(2*np.pi*f1*t) + 0.5*np.sin(2*np.pi*f2*t)
    return (sig + noise).astype('float32')

rng = np.random.default_rng(0)
X_audio, y_label = [], []
for k in range(3):
    for _ in range(N_PER_CLASS):
        X_audio.append(make_sample(k, rng)); y_label.append(k)
print('샘플 수:', len(X_audio))

## 2. 멜 스펙트로그램 변환

각 오디오를 멜 스펙트로그램으로 바꿔 CNN 입력(1채널 이미지)으로 만듭니다.

In [ ]:
def to_melspec(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=64, fmax=SR//2)
    return librosa.power_to_db(S, ref=np.max)

specs = np.stack([to_melspec(y) for y in X_audio])     # (N, 64, T)
specs = (specs - specs.mean()) / (specs.std() + 1e-9)  # 정규화
X = torch.tensor(specs).unsqueeze(1).float()           # (N, 1, 64, T)
yt = torch.tensor(y_label).long()
print('입력 shape:', X.shape)

# 학습/테스트 분할
idx = torch.randperm(len(X))
n_tr = int(0.8*len(X))
tr, te = idx[:n_tr], idx[n_tr:]

## 3. CNN 학습 및 평가

작은 CNN으로 멜 스펙트로그램을 분류합니다.

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(), nn.Linear(32*4*4, 64), nn.ReLU(), nn.Linear(64, n_classes),
        )
    def forward(self, x): return self.net(x)

model = AudioCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

Xtr, ytr = X[tr].to(device), yt[tr].to(device)
Xte, yte = X[te].to(device), yt[te].to(device)

for epoch in range(15):
    model.train(); opt.zero_grad()
    loss = crit(model(Xtr), ytr); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = (model(Xte).argmax(1) == yte).float().mean().item()
print(f'테스트 정확도: {acc*100:.1f}%')

### 정리

- 오디오를 멜 스펙트로그램으로 변환해 CNN으로 분류하는 표준 파이프라인을 구현했습니다.
- 합성 데이터로 개념을 명확히 봤습니다. 실제로는 UrbanSound8K, ESC-50 같은 데이터셋이나 직접 수집한 오디오를 사용합니다.
- 같은 구조로 음악 장르 분류, 환경음 인식, 명령어 분류 등에 응용할 수 있습니다.